# Evaluation of Synthetic Data Generation Methods for Tabular Health Data

## Models Evaluated

In this study we evaluate both probabilistic and generative synthetic data generation methods on tabular health data,
using implementations from **Synthetic Data Vault (SDV)** and **synthcity** Python libraries.

For each model we also try a differentially private (DP) version and later evaluate both.

### Probabilistic

1. **Gaussian Copulas**
    - `GaussianCopulaSynthesizer`
    - `DPGCSynthesizer` (DP version)
2. **Bayesian Networks**
    - `BayesianNetworkPlugin`
    - `PrivBayes` (DP version)

### Generative

1. **Generative Adversarial Networks**
    - `CTGAN`
    - DP variants: `DP-GAN`, `PATEGAN`, `AdsGAN`
2. **Variational Auto-Encoders**
    - `TVAE`, `RTVAE`
    - DP variants
3. **Normalizing Flows**
4. **Autoregressive / Tree-Based Models**
    - ARF (Adversarial Random Forests)
6. **Diffusion Models**
    - `TabDDPM` (DP)
    - `TableDiffusion` (DP)

## Data

A public clinical dataset **Heart Disease** from **UC Irvine Machine Learning Repository** is used in this study.

In [1]:
%pip install ucimlrepo

In [1]:
from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

X

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0
301,57,0,2,130,236,0,2,174,0,0.0,2,1.0,3.0


# Generative Models

## Generative Adversarial Network (GAN)

In this section we use GAN models to generate synthetic datasets from our original data.

### General Idea

There are two components - generator and discriminator. Generator creates a realistic synthetic row of tabular data with random noise applied. Discriminator acts as a binary classifier deciding whether the provided row is real or fake (0-1), taking either real or synthetic row as an input.

`[insert GAN loss function here]`

Discriminator (D) wants to classify correctly real samples (high probability) and fake samples (low probability).

Generator (G) wants to make Discrimator (D) classify fake samples as real.

### GAN on Tabular Data

Conditional Tabular Generative Adversarial Network (CTGAN) can explicitly condition the data generation process on specific categorical variables.

A typical CTGAN training loop:

1. Sample real batch from dataset.
2. Sample noise z.
3. Generator creates synthetic samples G(z).
4. Discriminator trains on:
    - real samples → label 1
    - fake samples → label 0
5. Generator trains to fool the discriminator:
    - update G so D(G(z)) → 1
6. Apply conditional constraints (CTGAN)
7. Repeat until convergence.

In [3]:
%pip install torch==2.2.2

In [11]:
%pip install synthcity

In [12]:
%pip install GReaT

In [6]:
%pip install opacus==1.5.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: opacus
    Found existing installation: opacus 1.5.4
    Uninstalling opacus-1.5.4:
      Successfully uninstalled opacus-1.5.4


In [2]:
from synthcity.plugins import Plugins

Plugins(categories=["generic", "privacy"]).list()

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


[2025-11-16T11:57:22.850635+0000][15954][CRITICAL] load failed: 
GReaT is not installed. Please install it with pip install GReaT.
Please be aware that GReaT is only available for python >= 3.9.

[2025-11-16T11:57:22.853192+0000][15954][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2025-11-16T11:57:22.854185+0000][15954][CRITICAL] module plugin_great load failed
[2025-11-16T11:57:22.859254+0000][15954][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2025-11-16T11:57:22.864842+0000][15954][CRITICAL] module disabled: /usr/local/lib/python3.12/dist-packages/synthcity/plugins/generic/plugin_goggle.py


['uniform_sampler',
 'adsgan',
 'decaf',
 'rtvae',
 'bayesian_network',
 'dpgan',
 'privbayes',
 'nflow',
 'ddpm',
 'tvae',
 'marginal_distributions',
 'pategan',
 'aim',
 'arf',
 'dummy_sampler',
 'ctgan']

In [4]:
X["target"] = y
X

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45,1,1,110,264,0,0,132,0,1.2,2,0.0,7.0,1
299,68,1,4,144,193,1,0,141,0,3.4,2,2.0,7.0,2
300,57,1,4,130,131,0,0,115,1,1.2,2,1.0,7.0,3
301,57,0,2,130,236,0,2,174,0,0.0,2,1.0,3.0,1


In [5]:
plugin = Plugins().get("ctgan", n_iter = 100)

[2025-11-16T12:18:50.779499+0000][15954][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2025-11-16T12:18:50.779499+0000][15954][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2025-11-16T12:18:50.782772+0000][15954][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2025-11-16T12:18:50.782772+0000][15954][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2025-11-16T12:18:50.784312+0000][15954][CRITICAL] module plugin_great load failed
[2025-11-16T12:18:50.784312+0000][15954][CRITICAL] module plugin_great load failed
[2025-11-16T12:18:50.785694+0000][15954][CRITICAL] module disabled: /usr/local/lib/python3.12/dist-packages/synthcity/plugins/generic/plugin_goggle.py
[2025-11-16T12:18:50.785694+0000][15954][CRITICAL] module disabled: /usr/local/lib/python3.12/dist-packages/synthcity/plugins/gener

### CTGAN plugin in synthcity

Most parameters of the plugin describe architectures or training loop of Generator (G) and Discriminator (D) neural networks.

#### Generator parameters

- `generator_n_layers_hidden`: how many hidden layers the generator neural network has (excluding input/out). More layers -> more capacity, but slower + risk of overfitting.
- `generator_n_units_hidden`: number of neurons per hidden layer. Controls generator model size.
- `generator_nonlin`: activation function used between layers:
    - `leaky_relu` - default, stable for GANs
    - `relu` - simpler
    - `elu` / `selu` - sometimes help with continuous skewed data
- `generator_dropout`: fraction of neurons randomly dropped during training. Prevents overfitting. Usually 0 for tabular GANs.
- `n_iter`: total number of training iterations (epochs). Higher -> better model (until overfitting), longer training.

#### Discriminator parameters

- `discriminator_n_layers_hidden`: how many hidden layers the discriminator has.
- `discriminator_n_units_hidden`: number of neurons per layer
- `discriminator_nonlin`: activation function for the discriminator.
- `discriminator_dropout`: dropout regularization for the discriminator.
- `discriminator_n_inter`: number of discriminator update steps per cycle.

#### Optimization and training

- `lr`: learning rate for both generator and discriminator
- `weight_decay`: L2 regularization on weights, which prevents exploding weights (and overfitting). Adds penalty to the loss function proportional to the sum of squares of the model's weights.
- `batch_size`: mini-batch size used during training. For a small dataset like in our case (303 rows), values 32-128 work good.
- `random_state`: random seed for reproducibility.
- `clipping_value`: gradient clipping threshold.

#### Preprocessing / encoding

- `encoder_max_clusters`: continuous variables are encoded using mode-specific normalization. This parameter controls the max number of mixture components (clusters) used. More clusters = better modeling of multimodal distributions, slower training.

In [7]:
plugin.fit(X)

100%|██████████| 100/100 [00:19<00:00,  5.09it/s]


In [8]:
plugin.generate(50)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,58,0,4,155,331,0,0,167,0,1.579432,1,2.0,3.0,0
1,59,0,3,150,314,0,2,165,0,1.523318,1,3.0,3.0,0
2,62,0,4,174,362,0,2,164,0,0.389325,2,2.0,3.0,1
3,61,0,3,154,329,0,2,164,0,1.585375,1,1.0,3.0,0
4,63,0,1,151,368,0,0,174,0,0.244118,2,2.0,3.0,1
5,59,1,1,185,321,0,2,178,0,0.177883,1,2.0,3.0,2
6,58,0,3,173,356,0,0,171,0,1.403261,2,2.0,3.0,0
7,59,0,4,160,382,0,0,172,0,1.766934,1,2.0,3.0,0
8,63,0,3,155,336,0,0,172,0,0.217632,1,0.0,3.0,0
9,56,0,3,148,318,0,0,163,0,0.302441,1,0.0,3.0,0
